# 1. Global UAP Sighting Patterns

## NUFORC Dataset: Initial Exploration and Preparation

This notebook focuses on the **National UFO Reporting Center (NUFORC)** dataset, which contains civilian-submitted reports of unusual or atmospheric observations. 

In recent government and scientific discussions, the term **UAP (Unidentified Anomalous Phenomena)** has replaced the older term **UFO**. The updated terminology reflects a broader scientific investigation into unexplained observations occurring in the **air, sea, or space**, and avoids the cultural stigma with the word "UFO". The focus of UAP research is on data collection, national security, and scientific analysis rather than assumptions about extraterrestrial origins. 

This project analyzes global sighting patterns using three primary datasets:
1. **Kaggle UFO Sightings Dataset (1906–2014)**  
   A large historical dataset of civilian-reported sightings.

2. **NUFORC Dataset (National UFO Reporting Center)**  
   A modern civilian reporting database containing detailed narrative descriptions of sightings.

3. **GEIPAN Dataset (Groupe d'Études et d'Informations sur les Phénomènes Aérospatiaux Non Identifiés)**  
   An official French government program responsible for investigating unidentified aerospace phenomena.

Each dataset has a different structure and level of detail. Because of these differences, each dataset is explored and cleaned in separate notebooks before being standardized and integrated into a unified relational database for analysis.

## 2. Purpose of This Notebook

The goal of this notebook is to perform an initial exploration and structural assessment of the NUFORC dataset before detailed cleaning and transformation. 

Specifically, this notebook will:
- Load the raw NUFORC dataset
- Inspect its structure, contents, and size of the dataset 
- Examine how sighting information is stored within the dataset.
- Identify key attributes embedded in the raw report text
- Define the fields that must be extracted and standardized

This notebook focuses on understanding the dataset structure and preparing a strategy for parsing the report data. Data extracted and transformation will occur in later steps of the project pipeline. 

## 3. Import Libraries

The following Python libraries are used for data loading, inspection, and early-stage preparation. 

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

## 4. Load NUFORC Dataset

The NUFORC file is stored in the project's 'data/raw/' directory. 

At this stage, the objective is simply to confirm that the dataset loads correctly and to preview the raw structure of the data. 

The dataset currently contains two columns: 
- **sighting** — a unique identifier for each reported event
- **report** — a narrative text field containing multiple attributes embedded in a single string
 
Because key information such as location, time, object shape, and duration are embedded within the narrative report text, additional parsing will be required to convert this data into structured columns suitable for database storage and analysis. 

In [23]:
# Files:
nuforc = pd.read_csv("../data/raw/nuforc_flat.csv")

nuforc.head()

,sighting,report
0,114864,Occurred: 2014-09-21 13:00:00 Local\nLocation:...
1,126755,Occurred: 2015-12-18 13:00:00 Local\nLocation:...
2,106946,Occurred: 2014-02-05 21:00:00 Local\nLocation:...
3,161419,Occurred: 2019-05-16 12:00:00 Local\nLocation:...
4,18735,Occurred: 2001-07-29 23:59:00 Local\nLocation:...


## 5. Initial Dataset Inspection

Before cleaning the dataset, I will inspect its size, column names, and a small random sample of rows. This will help determine how much preprocessing will be required and whether the dataset structure aligns with the schema used in this project.

### Dataset Dimensions

In [24]:
nuforc.shape

(147890, 2)

**Interpretation:** The dataset contains 147,890 records and 2 columns. This confirms that the NUFORC dataset stores most case information within a single narrative report field rather than in a fully structured tabular format.

### Column Names

In [25]:
nuforc.columns

Index(['sighting', 'report'], dtype='object')

**Interpretation:** The dataset contains only two fields: `sighting`, which functions as a case identifier, and `report`, which stores multiple case attributes inside a narrative text field. This indicates that additional parsing will be required before the dataset can be integrated into the project database.

### 5 Sample Records

In [26]:
nuforc.sample(5, random_state=42)

,sighting,report
45564,143911,Occurred: 2018-11-26 20:02:00 Local\nLocation:...
111642,178414,Occurred: 2023-09-20 14:28:00 Local\nLocation:...
54239,92690,Occurred: 2012-09-10 15:15:00 Local\nLocation:...
145758,121404,Occurred: 2015-08-22 23:40:00 Local\nLocation:...
68331,89799,Occurred: 2012-06-21 00:00:00 Local\nLocation:...


**Interpretation**: 
- The sample records show that observational details such as occurrence time, location, and descriptive narrative are embedded within the 'report' text field. 
- This confirms that processing will require text parsing to extract structured fields for analysis.

## 6. Raw Dataset Structure

The NUFORC dataset differs significantly from the historical Kaggle dataset used earlier in this project.  
Instead of storing each attribute in its own column, the NUFORC dataset stores most sighting details inside a single narrative text field.

At this stage, the dataset contains two columns:

1. **sighting** — a unique identifier for each reported event  
2. **report** — a narrative text field containing multiple case attributes embedded within the narrative text.

Important information such as observation date, time, location, object shape, duration, summary details, and descriptive narrative is embedded within the `report` field rather than stored as separate structured variables.

Because of this structure, the next stage of processing will involve parsing the `report` field into individual structured columns that can be standardized and integrated into a relational database. 

## 7. Data Preparation Plan

To integrate the NUFORC dataset with the project's relational database schema, several attributes must be extracted from the narrative `report` field.

Because the dataset stores multiple case details within a single text field, a parsing process will be used to identify and extract structured attributes where possible.

The following structured fields will be derived from the report text when available:

- observation_date  
- year  
- city  
- region_state_province  
- country  
- shape  
- duration_seconds  
- summary  
- description  
- latitude (when available)  
- longitude (when available)  

Geographic coordinates (`latitude` and `longitude`) may appear within some NUFORC report entries. When coordinates are present in the narrative text, they will be extracted during the parsing stage.

However, coordinates are not consistently available for all reports. For records where latitude and longitude are missing, geographic coordinates will be generated during a later geocoding step using the standardized location fields.

## 8. Next Steps

The next phase of the workflow will involve creating a custom parsing function to extract structured attributes from the NUFORC 'report' text. 

This parsing process will:

- identify patterns within the narrative report
- extract standardized fields
- convert text-based information into structured columns
- prepare the dataset for cleaning, standardization, and later integration into the project's relational database. 

Once the extraction process is complete, the structured NUFORC dataset will move to the cleaning and standardization stage before later integration with the other project datasets for cross-dataset analysis. 

### Inspect Common Report Labels

Before building the parser function, the labeled fields within the NUFORC `report` text are inspected. This helps determine which attributes appear consistently enough to be extracted reliably and which fields may require fallback logic or later enrichment.

In [27]:
common_labels = [
    "Occurred:",
    "Location:",
    "Shape:",
    "No of observers:",
    "Characteristics:",
    "Summary:",
    "Text:"
]

for label in common_labels:
    count = nuforc["report"].str.contains(label, regex=False, na=False).sum()
    print(f"{label} {count}")

Occurred: 147890
Location: 147889
Shape: 141568
No of observers: 141231
Characteristics: 106262
Summary: 146999
Text: 146999


### Observations

The inspection confirms that several labeled fields appear consistently within the NUFORC narrative reports.

- `Occurred` appears in essentially all records and can be used to extract the observation date.
- `Location` also appears on nearly all records, making it a reliable source for location parsing.
- `Shape`, `Summary`, and `Text` are also highly consistent and are strong candidates for structured extraction.
- `No of observers` and `Characteristics` appear in many records, but not all, so they should be treated as optional fields.

These findings confirm that the NUFORC `report` field follows a semi-structured format. Because several labels appear consistently, the parser can use them as anchors for extracting structured fields from the narrative text. 

### Parsing Function

The following function parses the NUFORC narrative 'report' field and extracts structured attributes such as date, location, shape, and duration when available. 

In [28]:
# re = Regular Expressions ("regex") module
# Used for pattern matching and extracting structured information from text
import re

def parse_nuforc_report(report_text):
    """
    Extract structured fields from the NUFORC report narrative.
    Returns a dictionary with extracted attributes.
    """

# pd.isna --> Pandas DataFrame, pd.isna = used to detect missing values for an array-like object
    if pd.isna(report_text):
        return {}

    result = {}

    # Extract observation date
    date_match = re.search(r"Occurred:\s*([^\n]+)", report_text)
    if date_match:
        result["observation_date"] = date_match.group(1).strip()

    # Extract location
    location_match = re.search(r"Location:\s*([^\n]+)", report_text)
    if location_match:
        result["location_raw"] = location_match.group(1).strip()

    # Extract shape
    shape_match = re.search(r"Shape:\s*([^\n]+)", report_text)
    if shape_match:
        result["shape"] = shape_match.group(1).strip()

    # Extract duration
    duration_match = re.search(r"Duration:\s*([^\n]+)", report_text)
    if duration_match:
        result["duration_raw"] = duration_match.group(1).strip()

    return result

**Interpretation:** 
- This initial parsing function focuses on the most consistently labeled fields identified in the previous inspection step. 
- It extracts observation date, location, shape, and duration from the narrative `report` text and stores them as structured key-value pairs for later transformation into columns.

### Test Parser

In [29]:
# Testing the parser on several reports
for i in range(5):
    print(parse_nuforc_report(nuforc["report"].iloc[i]))

{'observation_date': '2014-09-21 13:00:00 Local', 'location_raw': 'Huntsville, TX, USA', 'shape': 'Rectangle', 'duration_raw': 'several seconds'}
{'observation_date': '2015-12-18 13:00:00 Local', 'location_raw': 'Sonoma, CA, USA', 'shape': 'Sphere', 'duration_raw': '2 minutes'}
{'observation_date': '2014-02-05 21:00:00 Local', 'location_raw': 'Hershey, PA, USA', 'shape': 'Light', 'duration_raw': '10 seconds'}
{'observation_date': '2019-05-16 12:00:00 Local', 'location_raw': 'Brownsville, TX, USA', 'shape': 'Oval'}
{'observation_date': '2001-07-29 23:59:00 Local', 'location_raw': 'Tucson, AZ, USA', 'shape': 'Unknown', 'duration_raw': '?'}


**Note:** The parser is tested on several reports to confirm that the labeled fields are extracted correctly across multiple records. 

### Apply Parser

In [30]:
# Apply parser to dataset --> converts extracted values into a structured dataframe

parsed_reports = nuforc["report"].apply(parse_nuforc_report)

parsed_df = pd.json_normalize(parsed_reports)

parsed_df.head()

,observation_date,location_raw,shape,duration_raw
0,2014-09-21 13:00:00 Local,"Huntsville, TX, USA",Rectangle,several seconds
1,2015-12-18 13:00:00 Local,"Sonoma, CA, USA",Sphere,2 minutes
2,2014-02-05 21:00:00 Local,"Hershey, PA, USA",Light,10 seconds
3,2019-05-16 12:00:00 Local,"Brownsville, TX, USA",Oval,NaN
4,2001-07-29 23:59:00 Local,"Tucson, AZ, USA",Unknown,?


**Notes**: The parser is applied to the entire dataset, producing a structured dataframe containing extracted observation date, location, shape, and duration fields for each report.

### Structured Dataset

The parsed dataframe contains the structured fields extracted from the NUFORC narrative 'report' text. At this stage, the data remains in an intermediate format and has not yet been cleaned, standardized, or merged back into the original dataset. 

In [31]:
# Inspect the structured dataframe created from the parsed report text
display(parsed_df.head(7))
display(parsed_df.tail(7))

parsed_df.shape

,observation_date,location_raw,shape,duration_raw
0,2014-09-21 13:00:00 Local,"Huntsville, TX, USA",Rectangle,several seconds
1,2015-12-18 13:00:00 Local,"Sonoma, CA, USA",Sphere,2 minutes
2,2014-02-05 21:00:00 Local,"Hershey, PA, USA",Light,10 seconds
3,2019-05-16 12:00:00 Local,"Brownsville, TX, USA",Oval,NaN
4,2001-07-29 23:59:00 Local,"Tucson, AZ, USA",Unknown,?
5,2015-07-04 21:30:00 Local,"Scotch Plains, NJ, USA",Circle,1 hour
6,2002-01-07 17:45:00 Local,"Richmond, VA, USA",Light,approx 3 sec.


,observation_date,location_raw,shape,duration_raw
147883,2006-07-30 16:40:00 Local,"Sunderland (UK/England), , United Kingdom",Circle,10 - 15 Seconds
147884,2019-11-19 05:11:00 Local,"Sandy Springs, GA, USA",Light,2mns
147885,2020-07-12 23:00:00 Local,"Roseburg, OR, USA",Circle,30 seconds
147886,2010-08-04 21:57:00 Local,"Greensburg, IN, USA",Triangle,30 seconds
147887,2018-08-08 16:00:00 Local,"Boonville, MO, USA",Unknown,sitting in the car
147888,1997-10-01 08:00:00 Local,"Oregon (rural), OR, USA",Rectangle,20 seconds
147889,2011-01-13 20:05:00 Local,"Basye, VA, USA",Diamond,2 min


(147890, 4)

**Interpretation:**
- The structured dataframe confirms that the NUFORC report narratives were successfully parsed into structured fields.  
- The dataset now contains extracted observation date, location, object shape, and duration attributes for each report.
- Inspection of the first and last records confirms that the parser works consistently across the dataset.  
- The dataframe contains **147,890 parsed reports and 4 extracted fields**, which will be cleaned and standardized in the next stage of the analysis.

## 9. Data Cleaning

### Preserve Raw Parsed Data

Before applying cleaning transformations, a backup copy of the parsed NUFORC dataframe is created. This preserves the original extracted values and allows later cleaning steps to be tested without losing the intermediate parsed output.

In [32]:
# Create a backup copy of the parsed dataframe before cleaning
raw_backup_df = parsed_df.copy()

### Dataframe Structure and Data Types

In [33]:
# Inspect dataframe structure and data types
parsed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147890 entries, 0 to 147889
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   observation_date  147890 non-null  object
 1   location_raw      147889 non-null  object
 2   shape             141568 non-null  object
 3   duration_raw      140787 non-null  object
dtypes: object(4)
memory usage: 4.5+ MB


**Interpretation**: 
- Initial inspection shows that all extracted fields are currently stored as text rather than as cleaned analytical data types. 
- The `observation_date` field will need to be converted to a datetime format, while `duration_raw` will require normalization before it can be analyzed numerically.
- The output also shows that missing values are already present in some parsed fields, especially `shape` and `duration_raw`. 
- This confirms that additional cleaning steps will be necessary before the NUFORC data can be standardized and integrated into the final project dataset.

### Convert Observation Date to Datetime

The 'observation_date' field is currently stored as text. Converting it to a datetime format is the first cleaning transformation because it changes the field from a raw extracted string into a type that can be used for filtering, sorting, aggregation, and time-based analysis.

In [34]:
# Convert observation_date from text to datetime
parsed_df["observation_date"] = pd.to_datetime(
    parsed_df["observation_date"].str.replace(" Local", "", regex=False),
    errors="coerce"
)

**Interpretation**:

- This transformation converts the extracted observation date into a proper datetime format for analysis.
- Removing the trailing '" Local"' text allows Pandas to interpret the field consistently. Records that cannot be parsed will be converted to missing values and can be reviewed later.

### Check Missing Values

In [35]:
# Count missing values in each column
parsed_df.isna().sum()

observation_date     303
location_raw           1
shape               6322
duration_raw        7103
dtype: int64

**Interpretation:**
- Missing values were quantified across all fields to assess data completeness and identify areas requiring further cleaning.
- The 'observation_date' field now contains 303 missing values because those records failed to parse during datetime conversion. These entries likely contain irregular date formatting in the original NUFORC reports. 
- The 'location_raw' field contains only a single missing entry.
- However, the 'shape' and 'duration_raw' fields contain a larger number of missing values. This is expected because some NUFORC reports do not include these labeled attributes in their narratives. These fields will require additional handling during the data cleaning stage.

### Inspect Unique Values

In [37]:
# Inspecting shape
parsed_df["shape"].value_counts().head(20)

shape
Light        27496
Circle       14367
Triangle     13087
Other        10063
Unknown      10022
Fireball      9881
Disk          8716
Sphere        7652
Oval          6369
Orb           5924
Formation     4865
Changing      3987
Cigar         3753
Rectangle     2610
Cylinder      2482
Flash         2440
Diamond       2116
Chevron       1742
Egg           1289
Teardrop      1238
Name: count, dtype: int64

In [38]:
# Inspecting duration
parsed_df["duration_raw"].value_counts().head(20)

duration_raw
5 minutes      8813
2 minutes      6427
10 minutes     6188
1 minute       5531
3 minutes      4737
30 seconds     3966
15 minutes     3707
10 seconds     3382
5 seconds      2953
20 minutes     2748
30 minutes     2512
1 hour         2225
15 seconds     2042
20 seconds     1945
4 minutes      1647
3 seconds      1577
2 seconds      1226
2 hours        1116
2-3 minutes    1065
1-2 minutes     918
Name: count, dtype: int64

**Interpretation:**

- Categorical value distributions were analyzed to evaluate standardization potential and identify inconsistencies within the dataset.
- The 'shape' field contains many repeated categories, indicating strong potential for standardization and grouping in downstream analysis.
- The 'duration_raw' field includes common time expressions (e.g., minutes, seconds, hours), but also contains inconsistent formats such as ranges and abbreviated wording.
- These inconsistencies confirm that 'duration_raw' must be normalized into a numeric duration field before it can be used reliably in analysis.

### Normalize Duration to Seconds

-	The 'duration_raw' field contains text-based durations such as '"5 minutes"', '"30 seconds"', '"1 hour"', '"2-3 minutes"'. and '"approx 3 sec."'. 
-	To make duration analytically useful, these text values must be converted into a numeric duration field measured in seconds.

In [39]:
# Convert text-based duration values into numeric seconds
import re

def duration_to_seconds(duration_text):
    if pd.isna(duration_text):
        return pd.NA

    text = str(duration_text).strip().lower()

    # Handle obvious unknowns
    if text in ["?", "unknown", "unk", "na", "n/a", "NA", "Na", "NaN" "", "none"]:
        return pd.NA

    # Normalize wording
    text = text.replace("approx.", "approx")
    text = text.replace("approximately", "approx")
    text = text.replace("about", "")
    text = text.replace("around", "")
    text = text.replace("roughly", "")
    text = text.replace("approximately", "")
    text = text.replace(",", " ")
    text = text.strip()

    # Handle vague quantity words
    vague_patterns = [
        (r"\bcouple of\s+(second|seconds|sec)\b", 2),
        (r"\bcouple of\s+(minute|minutes|min)\b", 2 * 60),
        (r"\bcouple of\s+(hour|hours|hr|hrs)\b", 2 * 3600),

        (r"\bfew\s+(second|seconds|sec)\b", 3),
        (r"\bfew\s+(minute|minutes|min)\b", 3 * 60),
        (r"\bfew\s+(hour|hours|hr|hrs)\b", 3 * 3600),

        (r"\bseveral\s+(second|seconds|sec)\b", 5),
        (r"\bseveral\s+(minute|minutes|min)\b", 5 * 60),
        (r"\bseveral\s+(hour|hours|hr|hrs)\b", 5 * 3600),

        (r"\bhalf\s+(second|seconds|sec)\b", 0.5),
        (r"\bhalf\s+(minute|minutes|min)\b", 30),
        (r"\bhalf\s+(hour|hours|hr|hrs)\b", 1800),
    ]

    for pattern, seconds in vague_patterns:
        if re.search(pattern, text):
            return seconds

    # Handle mixed fractions like "2 1/2 hours"
    mixed_fraction_match = re.search(
        r"(\d+)\s+(\d+)/(\d+)\s*(second|seconds|sec|minute|minutes|min|hour|hours|hr|hrs)",
        text
    )
    if mixed_fraction_match:
        whole = float(mixed_fraction_match.group(1))
        numerator = float(mixed_fraction_match.group(2))
        denominator = float(mixed_fraction_match.group(3))
        value = whole + (numerator / denominator)
        unit = mixed_fraction_match.group(4)

    else:
        # Handle simple fractions like "1/2 hour"
        fraction_match = re.search(
            r"(\d+)/(\d+)\s*(second|seconds|sec|minute|minutes|min|hour|hours|hr|hrs)",
            text
        )
        if fraction_match:
            numerator = float(fraction_match.group(1))
            denominator = float(fraction_match.group(2))
            value = numerator / denominator
            unit = fraction_match.group(3)

        else:
            # Handle ranges like "2-3 minutes"
            range_match = re.search(
                r"(\d+(\.\d+)?)\s*-\s*(\d+(\.\d+)?)\s*(second|seconds|sec|minute|minutes|min|hour|hours|hr|hrs)",
                text
            )
            if range_match:
                low = float(range_match.group(1))
                high = float(range_match.group(3))
                value = (low + high) / 2
                unit = range_match.group(5)

            else:
                # Handle single values like "5 minutes", "30 seconds", "2 min", "1 hour"
                match = re.search(
                    r"(\d+(\.\d+)?)\s*(second|seconds|sec|minute|minutes|min|hour|hours|hr|hrs)",
                    text
                )
                if not match:
                    return pd.NA

                value = float(match.group(1))
                unit = match.group(3)

    # Convert to seconds
    if unit in ["second", "seconds", "sec"]:
        return value
    elif unit in ["minute", "minutes", "min"]:
        return value * 60
    elif unit in ["hour", "hours", "hr", "hrs"]:
        return value * 3600

    return pd.NA

#### Apply function

In [46]:
# Create numeric duration field in seconds
parsed_df["duration_seconds"] = parsed_df["duration_raw"].apply(duration_to_seconds)

In [47]:
# Count how many duration values were successfully converted
parsed_df["duration_seconds"].notna().sum()

np.int64(125431)

**Note**: This confirms the majority of duration values were successfully standardized into a usable numeric format.

In [48]:
# Count how many duration values remain missing after normalization
parsed_df["duration_seconds"].isna().sum()

np.int64(22459)

In [50]:
# Calculate percentage of missing duration values
total = len(parsed_df)
missing = parsed_df["duration_seconds"].isna().sum()
missing_pct = (missing / total) * 100

print(f"Missing duration valies: {missing} ({missing_pct:.2f}%")

Missing duration valies: 22459 (15.19%


**Interpretation**
- A total of 125,431 duration values were successfully converted into the numeric duration_seconds field.
- 22,459 values (15.19%) remain missing after normalization
- Missing values are expected, as many original duration_raw entries are blank, unknown, or too vague (e.g., “?”, “unknown”) to be reliably converted into a standardized numeric format.
- Despite this data loss, the normalization process retains a strong majority of usable observations, providing a solid foundation for downstream analysis involving duration.
- This transformation converts an unstructured narrative field into a quantitative variable suitable for statistical analysis.

### Data Quality Insights

In [53]:
parsed_df.loc[parsed_df["duration_seconds"].isna(), "duration_raw"].sample(20)

104706                         NaN
104665                         NaN
34085                        Still
21356                          NaN
119749                         NaN
36481                     Backyard
10795                  ten minutes
22023     Noticed after pics taken
101646                    2+ Hours
132885                           5
86205                        00:03
41969                          NaN
24125                          NaN
119681       Sense sunset the 15th
105638                Split second
104954      reflected until sunset
133363                 still there
139935                         NaN
29906                          NaN
7796                           NaN
Name: duration_raw, dtype: object

In [62]:
parsed_df.loc[parsed_df["duration_seconds"].isna(), "duration_raw"] \
    .value_counts().head(15)

duration_raw
seconds       492
unknown       492
Seconds       357
Unknown       330
?             236
10            218
15            186
30            180
ongoing       179
5             147
one minute    144
Ongoing       140
20            134
hours         127
1             117
Name: count, dtype: int64

**Interpretation:**
- Inspection of the remaining unparsed values shows that a significant portion are not valid duration expressions, but rather placeholders (e.g., “unknown”, “?”), incomplete entries (e.g., “seconds”, “hours”), or non-duration text (e.g., “ongoing”, narrative descriptions).
- Some entries (e.g., “10”, “15”, “30”) lack units, making it ambiguous whether they represent seconds, minutes, or hours, and therefore cannot be reliably standardized.
- While additional assumptions or heuristic rules could be introduced to reduce the percentage of missing values, doing so would risk introducing incorrect or misleading data.
- To preserve the integrity and reliability of the dataset, these ambiguous or invalid entries were intentionally left as missing rather than forcefully converted.
- The remaining ~15% of missing values reflects limitations in the original data quality and reporting consistency, rather than shortcomings in the parsing logic.
- Despite this, the normalization process successfully captures the majority of valid duration information, providing a strong and trustworthy foundation for downstream analysis.

### Validation of Duration Conversion

In [67]:
# Inspect a sample of original duration text alongside cleaned numeric seconds
parsed_df[["duration_raw", "duration_seconds"]].head(15)

,duration_raw,duration_seconds
0,several seconds,5.0
1,2 minutes,120.0
2,10 seconds,10.0
3,NaN,NaN
4,?,NaN
5,1 hour,3600.0
6,approx 3 sec.,3.0
7,1 minute,60.0
8,2 minutes,120.0
9,2 1/2 hours,9000.0


In [63]:
# Ensure the duration_seconds column is fully numeric
# Note: Any values that cannot be interpreted as numbers will be converted to NaN
parsed_df["duration_seconds"] = pd.to_numeric(
    parsed_df["duration_seconds"],
    errors="coerce"
)

In [ ]:
# Generate summary statistics (i.e., mean, median, quartiles, and max) for the cleaned numeric duration field
summary = parsed_df["duration_seconds"].describe()

# Extract key stats that are harder to interpret in raw seconds
mean_sec = summary["mean"]
std_sec = summary["std"]
max_sec = summary["max"]

# Helper function to convert seconds into minutes, hours, and days
def humanize(seconds):
    minutes = seconds / 60
    hours = seconds / 3600
    days = seconds / 86400
    return minutes, hours, days

# Convert the selected statistics into more readable units
mean_m, mean_h, mean_d = humanize(mean_sec)
std_m, std_h, std_d = humanize(std_sec)
max_m, max_h, max_d = humanize(max_sec)

# Print annotations - summary statistics 
print(f"Mean: {mean_sec:,.0f} seconds (~{mean_m:.1f} minutes)")
print(f"Std:  {std_sec:,.0f} seconds (~{std_h:.1f} hours)")
print(f"Max:  {max_sec:,.0f} seconds (~{max_d:.1f} days)")

Mean: 1,679 seconds (~28.0 minutes)
Std:  235,946 seconds (~65.5 hours)
Max:  82,800,000 seconds (~958.3 days)


In [68]:
# Flag extreme durations (greater than 24 hours)
parsed_df["is_extreme_duration"] = parsed_df["duration_seconds"] > 86400

# Count extreme values
extreme_count = parsed_df["is_extreme_duration"].sum()
extreme_pct = (extreme_count / len(parsed_df)) * 100

print(f"Extreme durations (>24h): {extreme_count} ({extreme_pct:.2f}%")

Extreme durations (>24h): 28 (0.02%


**Note**: For context, 86,400 seconds represents 24 hours (or 1 day), making it a practical threshold for identifying unusually long-duration observations.

In [69]:
parsed_df.loc[parsed_df["is_extreme_duration"], "duration_raw"].head(10)

4706           48 hours
7697       5010 minutes
8783         1:30 hours
14309           1700hrs
16548    72 hours (est)
28330            48 hrs
31300    36 hours or so
39339          55 hours
41494         .25 hours
42983        2400 hours
Name: duration_raw, dtype: object

**Interpretation:**
- This step converts the text-based `duration_raw` field into a numeric `duration_seconds` field that can be used in statistical analysis and visualization.
- This is one of the most important cleaning steps in the project because it transforms an inconsistent narrative-style field into a standardized quantitative measure.
- Missing or unclear duration values such as blank entries and "?" are intentionally preserved as missing values in `duration_seconds`, since they do not contain enough information to assign a reliable numeric duration.
- After conversion, 125,431 records were successfully interpreted as numeric durations, while the remaining records stayed missing because the original NUFORC reports either omitted duration entirely or used wording that could not be converted reliably.
- The summary statistics show that the distribution is strongly right-skewed: the mean duration is about 1,679 seconds (about 28 minutes), while the median is only 180 seconds (3 minutes). This indicates that a small number of extremely long-duration reports pull the average upward.
- The maximum parsed duration is extremely large (~958 days), indicating the presence of significant outliers.
- To systematically identify these extreme values, a flag (`is_extreme_duration`) was introduced for durations exceeding 86,400 seconds (24 hours), where 86,400 seconds represents one full day.
- Only 28 records (~0.02% of the dataset) exceed 24 hours, indicating that extreme duration values are extremely rare and do not materially impact the overall dataset.
- This suggests that while the maximum duration appears very large, it is driven by a negligible number of records rather than a widespread data quality issue.
- A brief inspection of these extreme cases shows entries such as "48 hours", "72 hours (est)", and "2400 hours", suggesting a mix of plausible long-duration sightings and potential reporting inconsistencies or exaggerations.
- As a result, these outliers are unlikely to distort most analyses but are still flagged for transparency and optional exclusion in downstream modeling or visualization.
- Rather than removing these values, they were identified, quantified, and structured for downstream analysis, preserving data integrity, while enabling flexible filtering. 

### Inspect Location Patterns

In [66]:
parsed_df["location_raw"].head(10)

0       Huntsville, TX, USA
1           Sonoma, CA, USA
2          Hershey, PA, USA
3      Brownsville, TX, USA
4           Tucson, AZ, USA
5    Scotch Plains, NJ, USA
6         Richmond, VA, USA
7       Youngstown, OH, USA
8         Bellevue, WA, USA
9             Dent, MN, USA
Name: location_raw, dtype: object

**Interpretation:**
- The sample location values indicate that the `location_raw` field generally follows a consistent pattern of "city, state, country", particularly for U.S.-based entries.
- This consistency supports the use of rule-based parsing to split the field into structured geographic components (e.g., city, state, country).
- However, this structure is not guaranteed across the entire dataset. International entries, missing components, and non-standard formatting (e.g., extra commas, abbreviations, or ambiguous place names) may introduce inconsistencies.
- While rule-based parsing is effective for the majority of records, additional validation and fallback handling may be required for edge cases.
- Overall, this step confirms that location standardization is feasible and appropriate, while highlighting the need for careful handling of non-standard entries in downstream processing.

In [80]:
# Count how many commas appear in each location string
# Goal: understand how consistently the data follows the expected format (i.e., "city, state, country" should have 2 commas)
parsed_df["location_raw"].str.count(",").value_counts()

location_raw
2.0    146461
3.0      1316
4.0       101
5.0         7
0.0         2
7.0         1
6.0         1
Name: count, dtype: int64

**Interpretation**:
- To assess how consistently the `location_raw` field follows the expected structure, the number of commas in each entry was analyzed.
- The majority of records (146,461) contain exactly two commas, which aligns with the expected format of "city, state/province, country".
- A smaller number of records contain additional commas, suggesting extra location details (e.g., neighborhoods, regions) or inconsistent formatting.
- A very small number of records contain fewer or no commas, indicating incomplete or non-standard entries.
- Overall, this confirms that the dataset is largely well-structured, but a subset of records will require additional handling during location standardization to ensure accurate parsing.

## 10. Location Standardization

In [ ]:
# Split the raw location string into structured geogrphic components 
location_split = parsed_df["location_raw"].str.split(",", expand=True)

# Assign the first three split components to city, state, and country
parsed_df["city"] = location_split[0].str.strip()
parsed_df["state"] = location_split[1].str.strip()
parsed_df["country"] = location_split[2].str.strip()

In [74]:
# Inspect results of location splitting
parsed_df[["location_raw", "city", "state", "country"]].head(15)

,location_raw,city,state,country
0,"Huntsville, TX, USA",Huntsville,TX,USA
1,"Sonoma, CA, USA",Sonoma,CA,USA
2,"Hershey, PA, USA",Hershey,PA,USA
3,"Brownsville, TX, USA",Brownsville,TX,USA
4,"Tucson, AZ, USA",Tucson,AZ,USA
5,"Scotch Plains, NJ, USA",Scotch Plains,NJ,USA
6,"Richmond, VA, USA",Richmond,VA,USA
7,"Youngstown, OH, USA",Youngstown,OH,USA
8,"Bellevue, WA, USA",Bellevue,WA,USA
9,"Dent, MN, USA",Dent,MN,USA


**Interpretation**:
- The location field was successfully parsed into structured geographic components (`city`, `state`, and `country`) using comma-based splitting.
- The sample output shows consistent and accurate extraction across records, with each component correctly aligned to its respective column.
- This confirms that the majority of entries follow the expected format of "city, state, country", particularly for U.S.-based records.
- The use of `.str.strip()` ensured that leading and trailing whitespace was removed, improving data cleanliness and consistency.
- While the sample indicates strong parsing performance, edge cases (e.g., international locations, missing components, or additional commas) may still exist and should be validated in aggregate.
- Overall, this step successfully transforms a semi-structured text field into structured geographic features suitable for filtering, grouping, and geographic analysis.

In [75]:
# Check missing values after splitting
parsed_df[["city", "state", "country"]].isna().sum()

city       1
state      3
country    3
dtype: int64

**Interpretation**:
- The number of missing values in the parsed location field is extremely low (i.e., missing = 1 city, 3 state, 3 country). This indicates that the majority of records were successfully structured.
- These missing values likely originate from incomplete or improperly formatted entries in the original 'location_raw' field, rather than errors in the parsing logic. 
- Given their negligible proportion relative to the full dataset, these records do not materially impact overall geographic analysis.
- Missing values are preserved rather than imputed or removed, thus maintaining data integrity and avoiding the introduction of assumptions.

In [77]:
# Validate how many components exist (handle missing values safely)
parsed_df["location_raw"].dropna().str.split(",").apply(len).value_counts()

location_raw
3    146461
4      1316
5       101
6         7
1         2
8         1
7         1
Name: count, dtype: int64

**Interpretation:**

- The distribution of component counts confirms that the overwhelming majority of records contain exactly three elements, aligning with the expected format of "city, state/province, country".
- A small number of records contain additional components (4+), likely representing extended location details such as neighborhoods, regions, or formatting inconsistencies.
- Very few records contain fewer components, indicating missing or incomplete geographic information.
- Overall, this validates that the dataset is highly structured and well-suited for rule-based parsing, with only minor edge cases requiring additional handling if higher precision is needed.

## 11. Geocoding

## 12. Exploratory Analysis

**Notes**: 